# Final Project: Implement a Jev-Like Classifier with ModernBERT


The final project turns the course's Jev-like scoring sketch into a runnable model. Implement a structured decision classifier from scratch around a pretrained ModernBERT encoder: given an input state, task instructions, and candidate descriptions, one shared scoring head must support changing label sets without adding a task-specific output class for each label. Train it on examples from multiple tasks, expose Choice, Noul, and Score operations, and measure transfer to a task held out from training.

This is an implementation of a published design pattern, not a recreation of Jev. The article says Jev's exact architecture, training data, and RLCD algorithm are proprietary and does not establish that Jev uses the ModernBERT scoring design. “From scratch” here means build the data adapter, candidate scorer, training and calibration pipeline, and inference interface yourself; initialize the encoder from a documented pretrained ModernBERT checkpoint rather than random weights.

## Project objective

Build a local, versioned service or Python package that accepts a state and one or more typed decision questions, then returns structured class probabilities or rubric scores. The system must work for more than one task using the same trained scoring head. A task is specified at inference time by natural-language instructions and candidate labels with short descriptions.

Use the [ModernBERT-base checkpoint](https://huggingface.co/answerdotai/ModernBERT-base) as the default backbone. ModernBERT is a pretrained encoder, not an autoregressive generator; it is suitable here because each candidate can be encoded together with the state and task instructions. Keep the run small enough to reproduce, and document if you freeze layers, use adapters, shorten sequences, subsample data, or train on a GPU.

## Data contract

Choose at least three text-classification tasks with different label spaces. A reasonable starter mix is IMDb sentiment, AG News topic classification, and Banking77 intent classification. Inspect each dataset's license, provenance, class balance, and official split before use. The article's IMDb result is a comparison reference, not a reason to reuse its reported test set as training data.

Convert every example to the same record shape:

```text
{
  "task_id": "sentiment",
  "instructions": "Determine the overall sentiment of the review.",
  "state": "The movie was ...",
  "candidates": [
    {"label": "negative", "description": "An overall unfavorable opinion."},
    {"label": "positive", "description": "An overall favorable opinion."}
  ],
  "gold_label": "positive"
}
```

Keep each task's train, validation, and test rows separate. Train the shared model on at least two tasks and reserve a third task entirely from model fitting for a task-transfer evaluation. Do not inspect that held-out task's test results to change prompts, descriptions, thresholds, or training settings. For an additional comparison, rotate which task is held out or train a task-specific baseline on it and clearly label that result as supervised on the target task.

## Model and training specification

Serialize a candidate-specific input from the task instructions, state, candidate label, and candidate description. Tokenize each candidate input with the ModernBERT tokenizer. Encode all candidates for an example, take the pooled `[CLS]` representation, and apply the same one-output linear head to every candidate. Let `h_i` be candidate *i*'s representation, `w` the shared weight vector, and `b` the bias. Compute its scalar score as `s_i = wᵀ h_i + b`, then normalize across the `K` candidates with `p_i = exp(s_i) / Σ_j exp(s_j)`.

Here `K` is the number of candidates for this question. Train on candidate groups with cross-entropy loss for the gold label. The output head has one scalar regardless of the number of labels, so the candidate list can change at inference time. The model's ability to handle new labels or tasks still has to be measured; the flexible interface alone does not create generalization.

Implement these operations on top of the shared scorer:

- **Choice:** score the supplied mutually exclusive candidates, normalize their scores with softmax, and return the selected label and per-label probabilities.
- **Noul:** represent one yes/no question with true and false candidates; return the true probability. Evaluate each question independently, so probabilities across multiple questions need not sum to one.
- **Score:** describe an ordered rubric as candidate levels, return the level probabilities, and map them to a numeric score using a rule you document, such as the probability-weighted level value.

Batch the candidates belonging to a decision when practical. Record maximum sequence length, truncation policy, tokenizer and checkpoint revision, trainable parameters, optimizer, learning rate, batch size, number of epochs, random seeds, and checkpoint-selection rule. Add a small CPU smoke path so another reader can check serialization and output shapes without repeating full fine-tuning.

## Calibration and evaluation

Use validation data for checkpoint and calibration choices; evaluate the held-out test partitions once after those choices are fixed. Compare the shared model with (1) the course's TF–IDF/logistic-regression baseline and (2) a ModernBERT classifier fine-tuned separately for each task where compute allows. Keep task-specific and cross-task results in separate rows.

Report accuracy and macro-F1 per task, plus confusion matrices and examples of consequential errors. Assess probabilities with log loss, Brier score, and a reliability plot or expected calibration error. Compare uncalibrated scores with validation-set temperature scaling. As an extension, add a Brier term to the training objective and measure whether it helps on held-out calibration data; do not assume the change improves calibration. Report mean and variation across at least three seeds when resources allow.

Measure end-to-end latency for one and several candidates, examples per second, peak memory, checkpoint size, and hardware. Include tokenization and batching choices in the measurement. For the held-out task, describe whether candidate wording alone was enough for useful transfer, which labels or examples failed, and what this test does not establish about general-purpose classification.

## Deliverables

1. **Reproducible implementation.** Data adapters, candidate serialization, shared ModernBERT scoring head, training entry point, inference functions, and a CPU smoke example. Separate reusable code from notebook analysis.
2. **Task and data record.** Dataset versions, licenses, provenance, task instructions, candidate descriptions, split IDs, class counts, and the task withheld from training.
3. **Model artifact and card.** Backbone revision, tokenizer, preprocessing, training configuration, seeds, selection rule, weights or download instructions, intended uses, and known limitations.
4. **Evaluation report.** Per-task metrics, calibration and threshold decisions, baselines, task-transfer results, errors, repeated-run variation, latency, memory, and checkpoint size.
5. **Decision interface.** Example requests and responses for Choice, Noul, and Score, plus input validation and clear probability semantics. State explicitly whether any returned confidence field is a class probability, a concentration summary, or another quantity.
6. **Bounded conclusion.** Identify what the model learned on its training tasks, whether it transferred to the held-out task, and which use cases still require a task-specific model or more evidence.

## Acceptance review

A reviewer must be able to run the small smoke path, inspect the shared one-scalar head, reproduce the candidate serialization, and trace each score to its task, candidate description, and model revision. The three API operations must return valid outputs, including for changed candidate sets and multiple independent Noul questions. Results must separate trained-task testing from the task withheld from training, compare probabilities as well as labels, and report operational cost. The write-up must call this a Jev-like ModernBERT implementation, not Jev, and must not present the article's reported IMDb numbers as results from this project.

## Optional extensions

- Compare cross-entropy-only training with cross-entropy plus a Brier penalty, then calibrate both on the same untouched validation split.
- Add a fourth task or rotate the held-out task to test how sensitive transfer is to the chosen domains.
- Serve the model behind a local JSON API and test malformed inputs, empty state, long inputs, and unknown question types.
- Compare a shared candidate scorer with conventional task-specific classification heads on the same target examples and compute budget.
